# Clamp Detection & Stable Tracking using YOLO11m + BoT-SORT

**Project objective:** detect 14 clamps mounted on a circular moving system, assign a stable logical ID from **1 to 14**, display the number of detected clamps, and expose a frame-level detection error percentage.

This notebook documents the full engineering workflow, including the failed approaches and the fixes that led to the final system.

## Stage 0 — Environment Setup
Install the main libraries used for training, inference, tracking, dataset download, and assignment.

In [ ]:
!pip install -q ultralytics roboflow scipy

In [ ]:
from ultralytics import YOLO
import cv2, os, math
import numpy as np
from scipy.optimize import linear_sum_assignment

print("Environment ready")

## Stage 1 — Dataset Strategy
The project was intentionally built using only **30 source images**. All images were manually reviewed so that adjacent clamps were annotated as separate objects whenever two clamps were visible next to each other.

Dataset split used in Version 3:
- Training: 24 images
- Validation: 4 images
- Testing: 2 images
- Classes: 1 (`Clamp`)
- Augmentation: OFF
- Preprocessing: Auto-Orient

## Stage 2 — Download Roboflow Dataset Version 3
Use the Roboflow-generated code for your project. Keep the API key private when sharing the notebook.

In [ ]:
from roboflow import Roboflow

# Replace with your own key when running.
rf = Roboflow(api_key="YOUR_API_KEY")
project = rf.workspace("zyad-khalaf-amen").project("clamp-detection-aqhg8")
version = project.version(3)
dataset = version.download("yolov11")

## Stage 3 — Locate `data.yaml`
Version 3 was downloaded to `Clamp-Detection-3`, so the correct training configuration is selected explicitly.

In [ ]:
data_yaml = "/kaggle/working/Clamp-Detection-3/data.yaml"
print("Using:", data_yaml)
print("Exists:", os.path.exists(data_yaml))

## Stage 4 — Train YOLO11m
YOLO11m was selected as the detector. Training was performed for 150 epochs at 960 px input resolution.

Final validation metrics obtained after training:
- Precision = **0.992**
- Recall = **0.982**
- mAP@50 = **0.995**
- mAP@50–95 = **0.827**

Because the validation set contains only 4 images, these metrics should be presented together with the dataset-size limitation.

In [ ]:
model = YOLO("yolo11m.pt")

train_results = model.train(
    data=data_yaml,
    epochs=150,
    imgsz=960,
    batch=4,
    device=0,
    optimizer="AdamW",
    lr0=0.001,
    patience=40,
    project="/kaggle/working/runs",
    name="clamp_v3_yolo11m"
)

## Stage 5 — Save the Best Weights

In [ ]:
import shutil
best_src = "/kaggle/working/runs/clamp_v3_yolo11m/weights/best.pt"
best_dst = "/kaggle/working/best.pt"
shutil.copy(best_src, best_dst)
print("Saved to:", best_dst)

## Stage 6 — Detection Test on the Original Video
Before tracking, the detector was tested alone. This separated **detection quality** from **tracking quality**.

In [ ]:
detector = YOLO("/kaggle/working/best.pt")
video = "/kaggle/input/datasets/zyadkhalafamen/clamp-video-test/Task.mp4"

results = detector.predict(
    source=video,
    conf=0.35,
    iou=0.50,
    imgsz=960,
    save=True,
    project="/kaggle/working",
    name="v3_detection_test",
    stream=True
)

for _ in results:
    pass

print("Detection finished")

## Problem 1 — Two Adjacent Clamps Were Detected as One
The first detector sometimes merged two neighboring physical clamps into one bounding box.

**Diagnosis:** this was a detector/annotation problem, not a tracker problem.

**Fix:** manually review all 30 annotations, ensure every physical clamp has its own box, create Dataset Version 3, and retrain YOLO11m.

**Status:** solved sufficiently for the tracking stage.

## Stage 7 — BoT-SORT Tracking
BoT-SORT was then added to preserve object identity between frames. Raw BoT-SORT IDs can become large values such as 129, 197, or 229; those values are internal tracker identities and are not suitable for the final presentation.

In [ ]:
tracker_yaml = "/kaggle/working/custom_botsort_v3.yaml"

tracked = detector.track(
    source=video,
    persist=True,
    tracker=tracker_yaml,
    conf=0.35,
    iou=0.50,
    imgsz=960,
    save=True
)

## Problem 2 — Raw IDs Were Large and Not User-Friendly
**Attempt 1:** remap tracker IDs to logical Clamp IDs 1–14.

This made the output easier to understand, but a new failure appeared: two detections could receive the same logical ID in the same frame.

## Problem 3 — Duplicate Logical IDs
Example: two boxes were both displayed as `Clamp 13`.

**Fix:** maintain a `used_ids` set for each frame so one logical ID can only be assigned once per frame.

**Result:** duplicate IDs in the same frame were removed.

In [ ]:
# Core idea used in the intermediate fix
used_ids = set()

# Before assigning a logical ID:
# if logical_id in used_ids: choose another unused ID
# after assignment:
# used_ids.add(logical_id)

## Problem 4 — Major ID Redistribution Around 22 Seconds
The intermediate system still failed during a strong detection drop around 22 s. When several detections disappeared and returned, nearest-position remapping reassigned many logical IDs incorrectly.

**Key insight:** the clamps are not moving randomly. They are constrained to a **circular path** and maintain their physical order. The final tracker should exploit that geometry.

## Stage 8 — Final Stable-ID Strategy
The final solution combines:
1. YOLO11m for clamp detection.
2. BoT-SORT as the temporal tracking layer.
3. A circular motion model for logical identity.
4. Hungarian one-to-one assignment between predicted logical clamp positions and current detections.
5. Angular gating to reject impossible jumps.
6. Motion prediction through temporary detection dropouts.

This prevents a short detector failure from causing a complete random redistribution of IDs.

In [ ]:
def normalize_angle(a):
    return (a + 2 * math.pi) % (2 * math.pi)

def angle_difference(a, b):
    d = abs(a - b)
    return min(d, 2 * math.pi - d)

def fit_circle(points):
    x = np.array([p[0] for p in points], dtype=float)
    y = np.array([p[1] for p in points], dtype=float)
    A = np.column_stack((2*x, 2*y, np.ones(len(points))))
    b = x*x + y*y
    c, _, _, _ = np.linalg.lstsq(A, b, rcond=None)
    return c[0], c[1]

## Stage 9 — Final Circular Tracking + Counter + Error Metric
Run the final stable-ID pipeline. The detector count is displayed as `Detected: n / 14` and the frame-level detection-count error is:

\[
Error(\%) = \frac{|14 - N_{detected}|}{14} \times 100
\]

This is a **count error**, not a complete MOT metric such as IDF1/HOTA/MOTA.

In [ ]:
model = YOLO("/kaggle/working/runs/clamp_v3_yolo11m/weights/best.pt")
input_video = "/kaggle/input/datasets/zyadkhalafamen/clamp-video-test/Task.mp4"
output_video = "/kaggle/working/V3_STABLE_CIRCULAR_IDS.mp4"
tracker_yaml = "/kaggle/working/custom_botsort_v3.yaml"
MAX_CLAMPS = 14

cap = cv2.VideoCapture(input_video)
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
writer = cv2.VideoWriter(output_video, cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height))

initialized = False
circle_cx = circle_cy = None
track_angles = {}
angular_velocity = {}
frame_number = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame_number += 1

    results = model.track(frame, persist=True, tracker=tracker_yaml,
                          conf=0.35, iou=0.50, imgsz=960, verbose=False)
    r = results[0]
    detections = []

    if r.boxes is not None:
        boxes = r.boxes.xyxy.cpu().numpy()
        confs = r.boxes.conf.cpu().numpy()
        for box, conf in zip(boxes, confs):
            x1, y1, x2, y2 = box
            detections.append({
                "box": box,
                "cx": (x1+x2)/2,
                "cy": (y1+y2)/2,
                "conf": float(conf)
            })

    detected_count = len(detections)

    if not initialized and detected_count == MAX_CLAMPS:
        centers = [(d["cx"], d["cy"]) for d in detections]
        circle_cx, circle_cy = fit_circle(centers)
        for d in detections:
            d["angle"] = normalize_angle(math.atan2(d["cy"]-circle_cy, d["cx"]-circle_cx))
        detections.sort(key=lambda d: d["angle"])
        for i, d in enumerate(detections):
            lid = i + 1
            track_angles[lid] = d["angle"]
            angular_velocity[lid] = 0.0
            d["logical_id"] = lid
        initialized = True

    elif initialized:
        predicted = {lid: normalize_angle(track_angles[lid] + angular_velocity[lid])
                     for lid in range(1, MAX_CLAMPS+1)}
        for d in detections:
            d["angle"] = normalize_angle(math.atan2(d["cy"]-circle_cy, d["cx"]-circle_cx))

        if detected_count > 0:
            logical_ids = list(range(1, MAX_CLAMPS+1))
            cost = np.zeros((MAX_CLAMPS, detected_count))
            for i, lid in enumerate(logical_ids):
                for j, d in enumerate(detections):
                    cost[i, j] = angle_difference(predicted[lid], d["angle"])

            rows, cols = linear_sum_assignment(cost)
            for row, col in zip(rows, cols):
                lid = logical_ids[row]
                if cost[row, col] > math.radians(25):
                    continue
                d = detections[col]
                d["logical_id"] = lid
                old = track_angles[lid]
                new = d["angle"]
                delta = new - old
                if delta > math.pi: delta -= 2*math.pi
                elif delta < -math.pi: delta += 2*math.pi
                angular_velocity[lid] = 0.85*angular_velocity[lid] + 0.15*delta
                track_angles[lid] = new

        assigned = {d["logical_id"] for d in detections if "logical_id" in d}
        for lid in range(1, MAX_CLAMPS+1):
            if lid not in assigned:
                track_angles[lid] = normalize_angle(track_angles[lid] + angular_velocity[lid])

    for d in detections:
        if "logical_id" not in d:
            continue
        x1, y1, x2, y2 = map(int, d["box"])
        lid = d["logical_id"]
        cv2.rectangle(frame, (x1,y1), (x2,y2), (0,255,0), 2)
        cv2.putText(frame, f"Clamp {lid} {d['conf']:.2f}", (x1, max(y1-6,20)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0,255,0), 1, cv2.LINE_AA)

    error = abs(MAX_CLAMPS-detected_count)/MAX_CLAMPS*100
    cv2.rectangle(frame, (20,20), (330,110), (0,0,0), -1)
    cv2.putText(frame, f"Detected: {detected_count} / 14", (35,55),
                cv2.FONT_HERSHEY_SIMPLEX, 0.75, (0,255,255), 2, cv2.LINE_AA)
    cv2.putText(frame, f"Error: {error:.1f}%", (35,88),
                cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0,255,255), 2, cv2.LINE_AA)
    if initialized and detected_count < 10:
        cv2.putText(frame, "DETECTION DROP", (width-400,60),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,0,255), 2, cv2.LINE_AA)
    writer.write(frame)

cap.release()
writer.release()
print("Saved:", output_video)

## Stage 10 — Final Evaluation Notes
### What was solved
- Adjacent clamps can be separated into individual detections.
- Logical IDs are constrained to 1–14.
- Duplicate logical IDs in the same frame are prevented.
- Temporary detection dropouts no longer force global random ID redistribution in the final circular strategy.
- The video displays the detected count and frame-level count error.

### Remaining limitations
- Only 30 source images were used.
- Validation contains only 4 images, so validation metrics can look optimistic.
- The displayed error percentage measures count error only; it does not fully measure localization or identity tracking quality.
- A stronger academic evaluation would add ID switches, IDF1/HOTA/MOTA and a larger independent test set.

## Engineering Iteration Summary
`30 images → YOLO11m → merged adjacent clamps → annotation review + retraining → good detection → BoT-SORT → large raw IDs → logical IDs 1–14 → duplicate ID failure → one-ID-per-frame constraint → detection drop near 22 s → random redistribution → circular motion prediction + Hungarian assignment → stable logical tracking + counter + error display`